## pipeline

[こちら](https://huggingface.co/learn/llm-course/en/chapter2/2)のチュートリアルを日本語で解説

`pipeline`関数はTransformersの機能を最も簡易的に実行できるAPIです
例えば`pipeline("タスク名")`のように最初に引数でタスク名を指定し、推論タスクを簡単に実行できるインスタンスが作成できます。

```python
classifier = pipeline("sentiment-analysis")
```

また、`pipeline(タスク名, model=モデル名)`のように`model`引数でモデルを指定できます（モデル名を指定しないと、"distilgpt2"という小さなGPTモデルが使用される）。

```python
classifier = pipeline("sentiment-analysis", model="distilgpt2")
```

pipelineはTokenizer、Model、Post Processing処理を内包しています。

![](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg)

これらの処理を個別に実行するケースは後ほど紹介します。
ここではpipelineで実行できる各タスクについて解説します。

### タスクの指定

`pipeline("タスク名")`のタスク名に以下のような文字列を指定することで、推論タスクを簡単に実行できるインスタンスが作成できます

- テキスト系
    - "text-generation"：テキスト生成（チャットボット）
    - "text-classification"：テキスト分類
    - "sentiment-analysis"：感情分析
    - "translation"：機械翻訳
    - "zero-shot-classification"：ゼロショット分類（未学習ラベルでの分類）
    - "feature-extraction"：テキストのベクトル表現を取得
    - "fill-mask"：マスクされたトークンの予測（穴埋め）
    - "ner"：固有表現認識
    - "question-answering"：質問応答（文脈に基づく回答抽出） → TransformerV5では使用できない
    - "summarization"：要約生成 → TransformerV5では使用できない
- 画像系
    - "image-to-text"：画像から説明文を生成
    - "image-classification"：画像分類
    - "object-detection"：物体検出
- 画像とテキスト混合
    - "image-text-to-text"：画像と質問テキストを入力して回答テキストを生成する（いわゆるVQAタスク）
- 音声系
    - "automatic-speech-recognition"：音声をテキストに変換
    - "audio-classification"：音声をカテゴリ分類する
    - "text-to-speech"：テキストを音声に変換


#### 感情分析

例えば"sentiment-analysis"（感情分析）は以下のように実行できます。

In [ ]:
# パイプライン実行例（感情分析）
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier("I've been waiting for a HuggingFace course my whole life.")

pipelineの推論は複数データをリスト指定して一括実行もできます。

In [ ]:
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!", "I'm neutral about this."]
)

#### ゼロショット分類

ラベル付けされていないテキストを分類

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

#### テキスト生成

テキスト生成（後に続く文章を予測する）を使用したい場合、`pipeline("text-generation")`のように指定できます。

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

以下のような引数を指定することで、返答のパターンを変えることができます

- "max_length"引数：入力＋出力を合わせたトークン数の上限（実用上は出力トークンの上限を指定する"max_new_tokens"の方が使いやすい）
- "num_return_sequences"引数：複数パターンのテキスト生成

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
)

#### マスクされたトークンの予測（穴埋め）

`top_k`引数で推論出力する予測候補の数を指定します

In [ ]:
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

#### 固有表現認識

In [ ]:
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

#### 質問応答（transformers v5系の代替例）

`question-answering`タスク名はv5系で利用できないため、ここでは`text-generation`で文脈つき質問を実行します。

In [ ]:
from transformers import pipeline

qa_generator = pipeline("text-generation")
prompt = (
    "Context: My name is Sylvain and I work at Hugging Face in Brooklyn.\n"
    "Question: Where do I work?\n"
    "Answer:"
)
qa_generator(prompt, max_new_tokens=20, do_sample=False)

#### 要約（transformers v5系の代替例）

`summarization`タスク名はv5系で利用できないため、ここでは`text-generation`に要約指示を与えて実行します。

In [ ]:
from transformers import pipeline

summarizer = pipeline("text-generation", model="distilgpt2")
text = """
America has changed dramatically during recent years. Not only has the number of 
graduates in traditional engineering disciplines such as mechanical, civil, 
electrical, chemical, and aeronautical engineering declined, but in most of 
the premier American universities engineering curricula now concentrate on 
and encourage largely the study of engineering science. As a result, there 
are declining offerings in engineering subjects dealing with infrastructure, 
the environment, and related issues, and greater concentration on high 
technology subjects, largely supporting increasingly complex scientific 
developments. While the latter is important, it should not be at the expense 
of more traditional engineering.

Rapidly developing economies such as China and India, as well as other 
industrial countries in Europe and Asia, continue to encourage and advance 
the teaching of engineering. Both China and India, respectively, graduate 
six and eight times as many traditional engineers as does the United States. 
Other industrial countries at minimum maintain their output, while America 
suffers an increasingly serious decline in the number of engineering graduates 
and a lack of well-educated engineers.
"""

prompt = f"Summarize the following text in 2 concise sentences:\n\n{text}\n\nSummary:"
summarizer(prompt, max_new_tokens=80, do_sample=False, return_full_text=False)

## pipelineを使用せずに個別処理する方法

pipelineはTokenizer、Model、Post Processing処理を内包しています。pipelineを使用せずに、これらの処理を個別実行する方法は次のノートブックで解説します。